In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [2]:
ds = load_dataset(
    "parquet",
    data_files="https://huggingface.co/datasets/ds-training-nba/nba_shot_data/resolve/main/raw_merged/merged_dataset.parquet"
)
df = ds["train"].to_pandas()

In [3]:
df.head()

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,...,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
0,4,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,0,...,0.0,NaN,1.610613e+09,Boston,Celtics,BOS,0.0,1996,False,NaN
1,6,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,15,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
2,7,PT11M06.00S,1,1610612738,BOS,133,Wesley,D. Wesley,163,122,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
3,43,PT11M06.00S,1,1610612741,CHI,893,Jordan,M. Jordan,45,148,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
4,8,PT11M02.00S,1,1610612738,BOS,677,Williams,E. Williams,0,0,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN


In [4]:
summary_data = []

for i, col in enumerate(df.columns):
    # Basic Info
    col_type = df[col].dtype
    rows_missing = df[col].isna().sum()
    missing_percent = (rows_missing / len(df.index)) * 100
    
    # Logic for Distribution Column
    distribution = ""
    
    # Check if the column is numerical
    if pd.api.types.is_bool_dtype(df[col]):
        desc = df[col].describe()
        # Formatting describe() output 
        distribution = (f"count: {desc['count']}, most frequent entry: {desc['top']}, amount: {desc['freq']}")
    elif pd.api.types.is_numeric_dtype(df[col]):
        desc = df[col].describe()
        # Formatting describe() output 
        distribution = (f"Mean: {desc['mean']:.2f}, Std: {desc['std']:.2f}, "
                        f"Min: {desc['min']}, 25%: {desc['25%']}, "
                        f"50%: {desc['50%']}, 75%: {desc['75%']}, Max: {desc['max']}")   
    # else categorical
    else:
        unique_values = df[col].unique()
        if len(unique_values) < 10:
            distribution = f"Unique values: {list(unique_values)}"
        else:
            distribution = f"{len(unique_values)} unique values"

    # Append row data
    summary_data.append({
        "#Column": i + 1,
        "Name of the Column": col,
        "Description": "",
        "Variable's type": str(col_type),
        "Rows with missing values": rows_missing,
        "Percentage of missing values": f"{missing_percent:.2f}%",
        "Categorical / Quantitative Distribution": distribution
    })

# Create the summary DataFrame
summary_df = pd.DataFrame(summary_data)

In [5]:
summary_df

,#Column,Name of the Column,Description,Variable's type,Rows with missing values,Percentage of missing values,Categorical / Quantitative Distribution
0,1,actionNumber,,int64,0,0.00%,"Mean: 271.96, Std: 169.09, Min: 0.0, 25%: 128...."
1,2,clock,,str,0,0.00%,1261 unique values
2,3,period,,int64,0,0.00%,"Mean: 2.53, Std: 1.14, Min: 1.0, 25%: 2.0, 50%..."
3,4,teamId,,int64,0,0.00%,"Mean: 1610611966.64, Std: 1124309.33, Min: 0.0..."
4,5,teamTricode,,str,4,0.00%,37 unique values
...,...,...,...,...,...,...,...
79,80,PLAYER3_TEAM_ABBREVIATION,,str,7483911,91.17%,37 unique values
80,81,VIDEO_AVAILABLE_FLAG,,float64,717,0.01%,"Mean: 0.36, Std: 0.48, Min: 0.0, 25%: 0.0, 50%..."
81,82,year,,int64,0,0.00%,"Mean: 2010.32, Std: 8.32, Min: 1996.0, 25%: 20..."
82,83,is_playoffs,,bool,0,0.00%,"count: 8208626, most frequent entry: False, am..."


In [6]:
# Export to Excel
summary_df.to_csv('../data/data_audit_generated_v2.csv', index=False)

# Additional manual checks

In [ ]:
df['PERIOD_x'].value_counts(normalize=True)*100

In [ ]:
df[['GAME_EVENT_ID', 'actionNumber']]
df['GAME_EVENT_ID'].equals(df['actionNumber'].astype('Int64'))

(df['GAME_EVENT_ID'] == df['actionNumber'].astype('Int64')).all()

In [ ]:
(df['SHOT_DISTANCE'] - df['shotDistance']).describe()

In [ ]:
df.loc[~df[['GAME_ID', 'GAME_EVENT_ID']].duplicated(keep=False), 'isFieldGoal'].value_counts()

In [ ]:
df[['isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal']].head(20)

In [ ]:
list(df['shotValue'].unique())
#df[['PERSON1TYPE', 'PLAYER1_NAME', 'PLAYER1_TEAM_CITY', 'shotValue']]
#df['NEUTRALDESCRIPTION'].dropna()

In [7]:
df.shape

(8208626, 84)